# 🎮 Video Game Sales Analysis Challenge  
You are a data analyst for a gaming industry publication. Your editor has a list of questions about historical sales data to prepare for an upcoming feature article. Use the vgsales.csv dataset and the techniques from the tutorial to find the answers.

---

## 0. Setup: Load the Data

First, make sure you have the DataFrame loaded, just as you did in the tutorial.

In [10]:
import pandas as pd
import numpy as np

# Load the dataset from the CSV file
df = pd.read_csv('vgsales.csv')

# df.head() # check if df loaded

## Part 1: Data Cleaning and Analysis

### Task 1: Data Cleaning and Preparation

Before you can analyze the data, you need to handle the missing values.

1. Drop Missing Publishers: Create a new DataFrame called df_cleaned that drops all rows where the Publisher is missing.

2. Fill Missing Years: In the df_cleaned DataFrame, use the forward-fill (.ffill()) method to fill the missing Year values.

3. Convert Year to Integer: After filling, convert the Year column from a float to an integer type.


#### **Task 1 Explanation:**

First I used dropna() on the read in df to drop null values, I specified the column using the subset parameter and set it to the Publisher column. Then I used the suggested forward-fill method to fill in missing year values. I only update the year column by specifying which column to update, and what column to use in the forward fill. Finally, I converted the the year back to an integer datatype using astype() because filling changed it to a float. 

In [11]:
# 1 drop missing publishers
df_cleaned = df.dropna(subset=['Publisher'])

# assert(df_cleaned['Publisher'].isna().sum() == 0) # makes sure there are 0 null values to ensure that empty publisher entries are dropped

# print(df_cleaned['Year'].isna().sum()) # before forward filling null count (249)
df_cleaned['Year'] = df_cleaned['Year'].ffill() # forward fill
# print(df_cleaned['Year'].isna().sum()) # after forward filling null count (0)

df_cleaned['Year'] = df_cleaned['Year'].astype('int64') # convert back to integer
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 16540 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16540 non-null  int64  
 1   Name          16540 non-null  str    
 2   Platform      16540 non-null  str    
 3   Year          16540 non-null  int64  
 4   Genre         16540 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16540 non-null  float64
 7   EU_Sales      16540 non-null  float64
 8   JP_Sales      16540 non-null  float64
 9   Other_Sales   16540 non-null  float64
 10  Global_Sales  16540 non-null  float64
dtypes: float64(5), int64(2), str(4)
memory usage: 1.5 MB


### Task 2: Answering Your Editor's Questions

Use the df_cleaned DataFrame for all analysis.

1. NumPy for Regional Analysis: What was the average (mean) sale in North America (NA_Sales) for games published by Activision?

    First, filter the DataFrame to only include games published by Activision.

    Then, extract the NA_Sales column as a NumPy array and use np.mean() to calculate the average.

2. Advanced Selection: How many games in the 'Role-Playing' genre sold more in Japan (JP_Sales) than in Europe (EU_Sales)?

3. Grouping and Aggregation: Which single platform has the highest total Global_Sales of all time?

4. Applying a Custom Function: Your editor wants to create a "Sales Skew" category. Create a new column called Sales_Skew.

    Write a function that takes a row as input.

    The function should return the string 'Japan-Skewed' if JP_Sales are greater than NA_Sales. Otherwise, it should return 'NA-Skewed'.

    Use the .apply() method to apply this function and create the new column.


#### **Task 2 Explanation:**

I filtered df_cleaned with .loc[] so only rows where publisher is activision remain. I took the NA_Sales column from that subset, converted it to a NumPy array using to_numpy(), and used .mean() on the array to get the average North American sales. For the Role-Playing question, I filtered again with .loc[] to keep only rows where Genre is Role-Playing. On that subset I compared JP_Sales and EU_Sales row by row, then used .sum() on the condition JP_Sales > EU_Sales to count how many games sold more in japan than in europe. To find the top platform, I grouped df_cleaned by platform with groupby(), summed Global_Sales for each platform, and used idxmax() to return the platform with the highest total sales. For Sales_Skew, I defined add_skewed() to return Japan-Skewed when JP_Sales is greater than NA_Sales and NA-Skewed otherwise. I copied df_cleaned to df_skew and used apply() with axis=1 to add the Sales_Skew column without changing the original DataFrame.

In [12]:
df_activision = df_cleaned.loc[df_cleaned['Publisher'] == 'Activision'] # filter for activision only

NA_sales = df_activision['NA_Sales'].to_numpy() # convert array
avg_NA_sales = NA_sales.mean()
print("Average NA Sales for games published by Activision are:", avg_NA_sales)

df_roleplaying = df_cleaned.loc[df_cleaned['Genre'] == 'Role-Playing'] # filter for role playing genres



count_jp_greater_than_eu = (df_roleplaying['JP_Sales'] > df_roleplaying['EU_Sales']).sum()
print("Role-Playing games with JP_Sales > EU_Sales:", count_jp_greater_than_eu)
# print("EU sales: ", df_roleplaying_eu)
# print("JP sales:", df_roleplaying_jp)

df_platform = df_cleaned.groupby('Platform')['Global_Sales'].sum().idxmax() # find the platform with the highest total Global Sales
print(df_platform, "has the highest total global sales of all time")

# function that returns "Japan-Skewed" or "NA-Skewed" based on "JP_Sales" and "NA_Sales" comparison
def add_skewed(row):
    if row["JP_Sales"] > row["NA_Sales"]:
        return "Japan-Skewed"
    else:
        return "NA-Skewed"

df_skew = df_cleaned.copy() # make a 1:1 copy of df_cleaned
df_skew["Sales_Skew"] = df_skew.apply(add_skewed, axis=1)
df_skew

Average NA Sales for games published by Activision are: 0.4407179487179488
Role-Playing games with JP_Sales > EU_Sales: 924
PS2 has the highest total global sales of all time


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Sales_Skew
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,NA-Skewed
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NA-Skewed
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,NA-Skewed
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,NA-Skewed
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NA-Skewed
...,...,...,...,...,...,...,...,...,...,...,...,...
16593,16596,Woody Woodpecker in Crazy Castle 5,GBA,2002,Platform,Kemco,0.01,0.00,0.00,0.00,0.01,NA-Skewed
16594,16597,Men in Black II: Alien Escape,GC,2003,Shooter,Infogrames,0.01,0.00,0.00,0.00,0.01,NA-Skewed
16595,16598,SCORE International Baja 1000: The Official Game,PS2,2008,Racing,Activision,0.00,0.00,0.00,0.00,0.01,NA-Skewed
16596,16599,Know How 2,DS,2010,Puzzle,7G//AMES,0.00,0.01,0.00,0.00,0.01,NA-Skewed


---
# Part 2: Data Combination and Selection

Setup: Create a New DataFrame for Reviews

Run the code below to create a new DataFrame. We'll pretend this data comes from a different source, like a game review aggregator.

In [13]:
# Create a new DataFrame with review scores
review_data = {
    'Name': ['Wii Sports', 'Super Mario Bros.', 'Grand Theft Auto V', 'Tetris', 'Red Dead Redemption 2'],
    'ReviewScore': [76, 90, 97, 88, 97]
}
reviews_df = pd.DataFrame(review_data)

### Task 3: Joining Tables with pd.merge

You need to combine the main sales data with the new review score data.

1. Inner Join: Find the review scores only for the games that exist in both your df_cleaned DataFrame and the new reviews_df.

2. Left Join: Add the ReviewScore to the top 10 best-selling games of all time from your original df_cleaned data. Some games may not have a review score, resulting in NaN values.

#### **Task 3 Explanation:**

First I used pd.merge() to combine df_cleaned and reviews_df with how set to inner and on set to name, so only games that appear in both tables are kept and I get review scores attached to the matching sales rows. For the left join, I built a subset of the ten highest selling games by sorting df_cleaned on Global_Sales in descending order and using head(10). I merged that subset with reviews_df using how left and the same name key, which keeps every row from the top 10 list and adds ReviewScore where a name exists in reviews_df, games without a match show as NaN in ReviewScore.

In [14]:
# Write your code here..
inner_join = pd.merge(df_cleaned, reviews_df, how="inner", on="Name") # inner join on 2 dfs on game names
inner_join

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,ReviewScore
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,76
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,90
2,6,Tetris,GB,1989,Puzzle,Nintendo,23.20,2.26,4.22,0.58,30.26,88
3,17,Grand Theft Auto V,PS3,2013,Action,Take-Two Interactive,7.01,9.27,0.97,4.14,21.40,97
4,24,Grand Theft Auto V,X360,2013,Action,Take-Two Interactive,9.63,5.31,0.06,1.38,16.38,97
5,45,Grand Theft Auto V,PS4,2014,Action,Take-Two Interactive,3.80,5.81,0.36,2.02,11.98,97
6,156,Tetris,NES,1988,Puzzle,Nintendo,2.97,0.69,1.81,0.11,5.58,88
7,198,Grand Theft Auto V,XOne,2014,Action,Take-Two Interactive,2.66,2.01,0.00,0.41,5.08,97
8,201,Super Mario Bros.,GB,1999,Platform,Nintendo,3.40,1.30,0.15,0.22,5.07,90
9,1898,Grand Theft Auto V,PC,2015,Action,Take-Two Interactive,0.36,0.64,0.00,0.08,1.08,97


In [15]:
top_10_global_sales = df_cleaned.sort_values("Global_Sales", ascending=False).head(10) # get the top 10 games based on global sales rankings
left_join = pd.merge(top_10_global_sales, reviews_df, how="left", on="Name") # left join on the top 10 games in global sales and the review_df on game name

left_join


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,ReviewScore
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,76.0
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,90.0
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,NaN
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,NaN
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN
5,6,Tetris,GB,1989,Puzzle,Nintendo,23.20,2.26,4.22,0.58,30.26,88.0
6,7,New Super Mario Bros.,DS,2006,Platform,Nintendo,11.38,9.23,6.50,2.90,30.01,NaN
7,8,Wii Play,Wii,2006,Misc,Nintendo,14.03,9.20,2.93,2.85,29.02,NaN
8,9,New Super Mario Bros. Wii,Wii,2009,Platform,Nintendo,14.59,7.06,4.70,2.26,28.62,NaN
9,10,Duck Hunt,NES,1984,Shooter,Nintendo,26.93,0.63,0.28,0.47,28.31,NaN


### Task 4: Slicing and Concatenating with .loc and pd.concat

You want to analyze sales data from different decades separately and then combine them again.

1. Slicing with .loc: Use the .loc[] accessor to create two new DataFrames:

    df_pre_2010: Should contain all games released before 2010.

    df_2010_plus: Should contain all games released in or after 2010.

2. Concatenation: Use pd.concat() to combine df_pre_2010 and df_2010_plus back into a single DataFrame.

3. Verification: As a final check, verify that the number of rows in the recombined DataFrame is equal to the number of rows in your df_cleaned DataFrame.

#### **Task 4 Explanation:**

I used .loc[] on df_cleaned with a boolean condition on year to split the data into two parts. df_pre_2010 keeps every row where year is less than 2010, and df_2010_plus keeps rows where year is greater than or equal to 2010, so each game falls into exactly one of the two frames. I then stacked them back together with pd.concat(), passing both dataframes in a list to rebuild one table. Finally, I used assert() to check that len(df_concat) equals len(df_cleaned), which confirms no rows were lost or duplicated when I sliced and concatenated.

In [16]:
# Write your code here..
df_pre_2010 = df_cleaned.loc[df_cleaned["Year"] < 2010] # filter df_cleaned to find entries with year > 2010
df_2010_plus = df_cleaned.loc[df_cleaned["Year"] >= 2010] # filter df_cleaned to find entries with year <= 2010

# df_pre_2010
# df_2010_plus

df_concat = pd.concat([df_pre_2010, df_2010_plus]) # join the dfs back together

assert (len(df_concat) == len(df_cleaned)) # returns an error message if the lengths don't match, otherwise, the concatenated df matches the original cleaned df